# Automated Job Applications
<br>

> #### Gino Prasad
> #### 01/29/2024
<br>


In [1]:
import selenium
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
import importlib
from collections import defaultdict
import urllib3
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
from datetime import datetime
import numpy as np
import re
from collections import defaultdict
from tqdm import tqdm
import subprocess as sp
from inspect import signature
import webbrowser
import zmq
import os, glob
from selenium.webdriver.support import expected_conditions as EC
import hashlib
import gensim.downloader
import utils
utils = importlib.reload(utils)
glove_vectors, common_words = utils.get_glove_vectors()

In [2]:
context = zmq.Context()
socket = context.socket(zmq.REP)
socket.bind("tcp://*:5555")

<SocketContext(bind='tcp://*:5555')>

In [45]:
logger = utils.Logger()
driver = None

In [4]:
def update():
    global utils
    global controller
    global GLOBALS
    driver = utils.GLOBALS["driver"]
    form_data = utils.GLOBALS["form_data"]
    initial_values = utils.GLOBALS["initial_values"]
    utils = importlib.reload(utils)

    utils.GLOBALS["logger"] = logger
    utils.GLOBALS["glove_vectors"] = glove_vectors
    utils.GLOBALS["common_words"] = common_words
    utils.GLOBALS["driver"] = driver
    utils.GLOBALS["form_data"] = form_data
    utils.GLOBALS["initial_values"] = initial_values
    
    GLOBALS = utils.GLOBALS
    for key in dir(utils):
        value = getattr(utils, key)
        if type(value) == type(lambda: None):
            globals()[key] = value
    
    controller = {
        'Restart Driver': utils.restart_driver,
        'Go To Link': utils.goto_link,
        'Click Button': utils.click_button,
        'Fill Input': utils.fill_input,
        'Print Buttons': utils.print_buttons,
        'Print Inputs': utils.print_inputs,
        'Close': utils.close,
        'Break': utils.break_,
    #     'Show Mouse': show_mouse, # Commented out in utils
    #     'Copy html': copy_html, # Commented out in utils
        'Next': utils.next_,
        'Record State': utils.record_state,
        'Broadcom Test': utils.open_broadcom,
        'Reset Form Log': utils.reset_form_log,
        'Get Predicted Values': utils.print_predicted_values,
        'Fill Predicted Values': utils.fill_predicted_values,
        'Update': update
    }

    with open("/Users/ginoprasad/Job_Applications/web_crawler/controller.txt", 'w') as outfile:
        for name, function in controller.items():
            num_parameters = len(signature(function).parameters.items())
            outfile.write(f"{name},{num_parameters}\n")
    return 'Updated Utils'
update()

'Updated Utils'

In [40]:
update()
while True:
    command = socket.recv().decode()
    if not command:
        logger.log("RECEIVED NOTHING")
        continue
    logger.log(command)
    command = command.split(',')
    try:
        output = str(controller[command[0]](*command[1:]))
    except KeyboardInterrupt as e:
        socket.send(b'')
        break
    except Exception as e:
        logger.log(f"ERROR FOUND {e.__class__}")
        output = f'FAILED REQUEST {e.__class__}'
    socket.send(output.encode())
    if ','.join(command) == 'Break':
        break

In [46]:
update()
utils.close()

''